# Run an AI model on Colab

Downloads an open model to this runtime's GPU and chats with it.

**How to run:** *Runtime → Change runtime type → T4 GPU* (or L4 / A100 with Colab Pro), then
*Runtime → Run all*. The first run downloads the model (~8 GB), which takes a few minutes.
At the bottom there's a chat box — type a message, or `quit` to stop.


In [ ]:
# Which GPU did we get? (If this errors: Runtime > Change runtime type > pick a GPU.)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
!pip install -q -U transformers accelerate


## Pick a model

Any chat model on [Hugging Face](https://huggingface.co/models?pipeline_tag=text-generation)
works if it fits in GPU memory — roughly **2 GB per billion parameters** at this precision.
A T4 has 15 GB, an L4 22 GB, an A100 40 GB.

- `Qwen/Qwen3-4B-Instruct-2507` — default; good quality, fits a T4
- `Qwen/Qwen3-1.7B` — smaller and faster
- Gated models (Llama, Gemma) need you to accept their license on Hugging Face and add an
  `HF_TOKEN` secret (🔑 panel) first.


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# T4s have no fast bfloat16, so use float16 there; newer GPUs get bfloat16.
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype, device_map="auto")
model.eval()
print(f"Loaded {MODEL_ID} — {torch.cuda.memory_allocated() / 1e9:.1f} GB on GPU")


In [ ]:
history = []

def chat(message, max_new_tokens=512, temperature=0.7):
    """Send a message, print the reply, and remember the conversation."""
    history.append({"role": "user", "content": message})
    inputs = tokenizer.apply_chat_template(
        history, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
            temperature=temperature, top_p=0.9,
        )
    reply = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": reply})
    print(reply)

def reset():
    history.clear()


In [ ]:
chat("In two sentences, what is a GPU and why do AI models need one?")


## Chat

Type below and press Enter. `reset` clears the conversation, `quit` stops.


In [ ]:
while True:
    message = input("you> ").strip()
    if message.lower() in ("quit", "exit", "q"):
        break
    if message.lower() == "reset":
        reset(); print("(conversation cleared)"); continue
    if message:
        print("model> ", end="")
        chat(message)
        print()
